# QC Linked-Read Barcode Summary

In [ ]:
indir <- "logs/bxcount"

In [ ]:
cat(format(Sys.time(), '🗓️ %d %B, %Y 🕔 %H:%M'))

In [ ]:
library(dplyr)
library(DT)
library(highcharter)
library(scales)
library(tidyr)

# Overview
##
<h2> General Per-Sample Barcode Statistics </h2>

This report details the counts of valid and invalid linked-read barcodes in the 
sample read headers. Validation varies between linked-read technologies; expand
the technologies below for an explanation of barcode format and what an invalidation
would look like. More exhaustive explanations can be found in 
[Harpy's documentation](https://pdimens.github.io/harpy/getting_started/linked_read_data/#linked-read-data-types).

<details>
<summary>Haplotagging</summary>
The correct haplotagging format is `BX:Z:AXXCXXBXXDXX`, where 
there is a tab before (and possibly after) `BX:Z` and `XX` are 2-digit numbers
between `00` and `96`. Additional tags (e.g. `QX:Z` or `VX:i`) are fine as long as they conform to the 
[SAM comment spec (section 1.5)](https://samtools.github.io/hts-specs/SAMv1.pdf) 
of `TAG:TYPE:VALUE`.

![haplotagging FASTQ record](https://pdimens.github.io/harpy/static/lr_haplotagging.svg)

</details>

<details>
<summary>stLFR</summary>
The correct stLFR format is `#X_Y_Z` at the end of the sequence ID, where `X`, `Y`, and `Z` are
integers of any value, with `0` being considered invalid.

![stLFR FASTQ record](https://pdimens.github.io/harpy/static/lr_stlfr.svg)

</details>

<details>
<summary>TELLseq</summary>
The correct TELLseq format is `:ATCGN` at the end of the sequence ID, where `ATCGN` are
uppercase nucleotides (`A`, `T`, `C`, `G`, `N`), with `N` being considered invalid. TELLseq
uses 18bp barcodes, so the barcodes are generally expected to be 18bp long and the presence
of `N` would make the barcode considered invalid.

![TELLseq FASTQ record](https://pdimens.github.io/harpy/static/lr_tellseq.svg)

</details>

In [ ]:
indir <- indir

infiles <- list.files(indir, ".count.log", full.names = TRUE)
for (j in seq_along(infiles)) {
  i <- infiles[j]
  samplename <- gsub(".count.log", "", basename(i))
  s_df <- read.table(i, header = F)
  total <- s_df$V2[1]
  bc_total <- s_df$V2[2]
  bc_ok <- s_df$V2[3]
  bc_invalid <- s_df$V2[4]
  if (bc_total == 0) {
    bc_percent <- 0
    ig_percent <- 0
  } else {
    bc_percent <- round((bc_ok / bc_total) * 100, digits = 3)
    ig_percent <- round((bc_invalid / bc_total) * 100, digits = 3)
  }
  if(j == 1){
      BXstats <- data.frame(
        Sample = samplename,
        TotalReads = total,
        TotalBarcodes = bc_total,
        ValidBarcodes = bc_ok,
        ValidPercent = bc_percent,
        InvalidBarcodes = bc_invalid,
        InvalidPercent = ig_percent
      )
      .invalid <- t(rbind(c("Sample", samplename), c("TotalBarcodes", bc_total),s_df[5:nrow(s_df),]))
      invalids <- data.frame(t(.invalid[2,]))
      names(invalids) <- .invalid[1,]
      invalids <- invalids %>% mutate_at(-1, as.numeric)
  } else {
    BXstats <- rbind(
      BXstats,
      data.frame(
        Sample = samplename,
        TotalReads = total,
        TotalBarcodes = bc_total,
        ValidBarcodes = bc_ok,
        ValidPercent = bc_percent,
        InvalidBarcodes = bc_invalid,
        InvalidPercent = ig_percent
      )
    )
    .invalid <- t(rbind(c("Sample", samplename), c("TotalBarcodes", bc_total), s_df[5:nrow(s_df),]))
    .inv <- data.frame(t(.invalid[2,]))
    names(.inv) <- .invalid[1,]
    .inv <- .inv %>% mutate_at(2:ncol(.inv), as.numeric)
    invalids <- rbind(invalids, .inv)
    }
}

figheight <- 3 + (0.6 * nrow(BXstats))

##

In [ ]:
list(
  icon = "files",
  color = "light",
  value = prettyNum(nrow(BXstats), big.mark = ",")
)

In [ ]:
list(
  icon = "check2-circle",
  color = "#dde6d5",
  value = min(BXstats$ValidPercent)
)

In [ ]:
list(
  icon = "check2-circle",
  color = "#dde6d5",
  value = max(BXstats$ValidPercent)
)

In [ ]:
list(
  icon = "check2-circle",
  color = "#dde6d5",
  value = round(mean(BXstats$ValidPercent),3)
)

In [ ]:
list(
  icon = "check2-circle",
  color = "#dde6d5",
  value = round(sd(BXstats$ValidPercent),3)
)

In [ ]:
absent <- BXstats$TotalReads-BXstats$TotalBarcodes
list(
  icon = "question-circle",
  color = "#f6e698ff",
  value = min(absent)
)

In [ ]:
list(
  icon = "question-circle",
  color = "#f6e698ff",
  value = max(absent)
)

## persamp
<h3> General Barcode Statistics </h3>
Below is a table listing all the samples Harpy processed and their associated
barcode statistics as determined by the sequences in the **forward-read file (R1) only**.
If for some reason `TotalBarcodes` equals `0`, then there may be an issue with
the format of your FASTQ headers.

In [ ]:
plotdata <- BXstats %>%
  select(Sample, ValidBarcodes, InvalidBarcodes) %>% 
  rename(Valid = ValidBarcodes, Invalid = InvalidBarcodes)

percdata <- BXstats %>%
  select(Sample, ValidPercent, InvalidPercent) %>% 
  rename(Valid= ValidPercent, Invalid = InvalidPercent)

##
###
::: {.card}

In [ ]:
hs <- hist(percdata$Valid, breaks = seq(0, 100, 5), plot = F)
hs <- data.frame(val = hs$breaks[-1], freq = hs$counts)

highchart() |>
  hc_chart(type = "area", animation = F) |>
  hc_add_series(data = hs, type = "areaspline", hcaes(x = val, y = freq), color = "#90cb8cff", name = "Valid Barcodes", marker = list(enabled = FALSE)) |>
  hc_title(text = "Distribution of Percent Valid Barcodes") |>
  hc_xAxis(title = list(text = "Percent Valid"), type = "linear") |>
  hc_yAxis(title = list(text = "Number of Samples")) |>
  hc_tooltip(crosshairs = TRUE, animation = FALSE,
    formatter = JS("function () {return '<b>' + this.series.name + '</b><br><b>' + this.x + '% valid</b><br/>Samples: <b>' + this.y + '</b>';}") 
  ) |>
  hc_exporting(enabled = T, filename = "valid.barcodes.distribution",
    buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
)

:::

###
::: {.card}

In [ ]:
column_description <- c(
  "name of the sample",
  "total number of reads",
  "total number of barcodes present",
  "number of valid linked-read barcodes",
  "percent of valid linked-read barcodes"
  )

headerCallback <- c(
  "function(thead, data, start, end, display){",
  paste0(" var tooltips = ['", paste(column_description, collapse = "','"), "'];"),
  "  for(var i=0; i<tooltips.length; i++){",
  "    $('th:eq('+i+')',thead).attr('title', tooltips[i]);",
  "  }",
  "}"
)

DT::datatable(
  BXstats[,1:5],
  rownames = FALSE,
  filter = "top",
  extensions = "Buttons",
  caption = "Overall barcode quality per sample",
  fillContainer = T,
  options = list(
    dom = "Brtp",
    buttons = c("csv"),
    scrollX = TRUE,
    headerCallback = JS(headerCallback)
  )
)

:::

# General Invalidations
## table desc
<h2> Percent Barcode Invalidations by Position </h2>
Below is information describing linked-read barcode invalidations across all **forward reads**
within samples. The term "positions" refers to:

| linked-read type | definition                                               |
|:-----------------|:---------------------------------------------------------|
| haplotagging     | the `A`, `B`, `C`, `D` segments of haplotagging barcodes |
| stLFR            | the `X`,`Y`,`Z` segments of stFLR barcodes               |
| TELLseq          | `N` nucleotide positions in TELLseq barcodes             |

##
###

In [ ]:
invalids_percent <- invalids %>% mutate_at(c(-1,-2), function(x){round(x/.$TotalBarcodes * 100, 3)})
invalids_percent$Sample <- as.factor(invalids$Sample)
invalids_long <- pivot_longer(invalids_percent, -1, names_to = "Position", values_to = "Count")
invalids_long$Position <- as.factor(paste0("position " , invalids_long$Position))

::: {.card}
This figure shows the distribution of percent invalid barcode positions
across all samples. This should help you assess whether some positions have a 
higher incidence of failure than others.

In [ ]:
position_names <- names(invalids)[3:ncol(invalids)]
# a 256-color distribution of the Turbo palette
# from https://github.com/dbaranger/turbo_palette_R/blob/main/turbo_hex.txt
palette <- c("#30123B","#321543","#33184A","#341B51","#351E58","#36215F","#372466","#38276D","#392A73","#3A2D79","#3B2F80","#3C3286","#3D358B","#3E3891","#3F3B97","#3F3E9C","#4040A2","#4143A7","#4146AC","#4249B1","#424BB5","#434EBA","#4451BF","#4454C3","#4456C7","#4559CB","#455CCF","#455ED3","#4661D6","#4664DA","#4666DD","#4669E0","#466BE3","#476EE6","#4771E9","#4773EB","#4776EE","#4778F0","#477BF2","#467DF4","#4680F6","#4682F8","#4685FA","#4687FB","#458AFC","#458CFD","#448FFE","#4391FE","#4294FF","#4196FF","#4099FF","#3E9BFE","#3D9EFE","#3BA0FD","#3AA3FC","#38A5FB","#37A8FA","#35ABF8","#33ADF7","#31AFF5","#2FB2F4","#2EB4F2","#2CB7F0","#2AB9EE","#28BCEB","#27BEE9","#25C0E7","#23C3E4","#22C5E2","#20C7DF","#1FC9DD","#1ECBDA","#1CCDD8","#1BD0D5","#1AD2D2","#1AD4D0","#19D5CD","#18D7CA","#18D9C8","#18DBC5","#18DDC2","#18DEC0","#18E0BD","#19E2BB","#19E3B9","#1AE4B6","#1CE6B4","#1DE7B2","#1FE9AF","#20EAAC","#22EBAA","#25ECA7","#27EEA4","#2AEFA1","#2CF09E","#2FF19B","#32F298","#35F394","#38F491","#3CF58E","#3FF68A","#43F787","#46F884","#4AF880","#4EF97D","#52FA7A","#55FA76","#59FB73","#5DFC6F","#61FC6C","#65FD69","#69FD66","#6DFE62","#71FE5F","#75FE5C","#79FE59","#7DFF56","#80FF53","#84FF51","#88FF4E","#8BFF4B","#8FFF49","#92FF47","#96FE44","#99FE42","#9CFE40","#9FFD3F","#A1FD3D","#A4FC3C","#A7FC3A","#A9FB39","#ACFB38","#AFFA37","#B1F936","#B4F836","#B7F735","#B9F635","#BCF534","#BEF434","#C1F334","#C3F134","#C6F034","#C8EF34","#CBED34","#CDEC34","#D0EA34","#D2E935","#D4E735","#D7E535","#D9E436","#DBE236","#DDE037","#DFDF37","#E1DD37","#E3DB38","#E5D938","#E7D739","#E9D539","#EBD339","#ECD13A","#EECF3A","#EFCD3A","#F1CB3A","#F2C93A","#F4C73A","#F5C53A","#F6C33A","#F7C13A","#F8BE39","#F9BC39","#FABA39","#FBB838","#FBB637","#FCB336","#FCB136","#FDAE35","#FDAC34","#FEA933","#FEA732","#FEA431","#FEA130","#FE9E2F","#FE9B2D","#FE992C","#FE962B","#FE932A","#FE9029","#FD8D27","#FD8A26","#FC8725","#FC8423","#FB8122","#FB7E21","#FA7B1F","#F9781E","#F9751D","#F8721C","#F76F1A","#F66C19","#F56918","#F46617","#F36315","#F26014","#F15D13","#F05B12","#EF5811","#ED5510","#EC530F","#EB500E","#EA4E0D","#E84B0C","#E7490C","#E5470B","#E4450A","#E2430A","#E14109","#DF3F08","#DD3D08","#DC3B07","#DA3907","#D83706","#D63506","#D43305","#D23105","#D02F05","#CE2D04","#CC2B04","#CA2A04","#C82803","#C52603","#C32503","#C12302","#BE2102","#BC2002","#B91E02","#B71D02","#B41B01","#B21A01","#AF1801","#AC1701","#A91601","#A71401","#A41301","#A11201","#9E1001","#9B0F01","#980E01","#950D01","#920B01","#8E0A01","#8B0902","#880802","#850702","#810602","#7E0502","#7A0403") 
selected_colors <- palette[seq(1,length(palette),length(palette)%/%(length(position_names)-1))]

In [ ]:
percent_breaks <- seq(0, 100, 5)
total_barcodes <- sum(invalids_percent$TotalBarcodes)
inv_hist <- data.frame(Position = character(), Bin = numeric(), Count = numeric())
for(pos in position_names){
  h <- hist(invalids_percent[,pos], breaks = percent_breaks, plot = F)
  inv_hist <- rbind(
    inv_hist,
    data.frame(Position = rep(paste("position", pos), length(percent_breaks)-1), Bin = h$breaks[-1], Count = h$count)
  )
}

hchart(inv_hist, "areaspline", hcaes(x = Bin, y = Count, group = Position), stack = "overlap", marker = list(enabled = FALSE)) |>
  hc_colors(selected_colors) |>
  hc_title(text = "Distribution of Percent Invalid Barcode Positions") |>
  hc_xAxis(title = list(text = "Percent Invalid"), type = "linear", max = 100) |>
  hc_yAxis(title = list(text = "Number of Samples")) |>
  hc_tooltip(crosshairs = TRUE, animation = FALSE,
    formatter = JS("function () {return '<b>' + this.series.name + '</b><br><b>' + this.x + '% invalid</b><br/>Samples: <b>' + this.y + '</b>';}") 
  ) |>
  hc_exporting(enabled = T, filename = "invalid.barcodes.distribution",
    buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
)


#if (length(unique(invalids_long$Count)) > 2) {
#  
#  inv_hist <- data.frame(Position = character(), Bin = numeric(), Count = numeric())
#  inv_breaks <- hist(invalids_long$Count, plot = F)
#  inv_breaks <- inv_breaks$breaks
#  for(j in position_names){
#    h <- hist(invalids_percent[, j], inv_breaks, plot = F)
#    position <- paste0("position ", j)
#    inv_hist <- rbind(inv_hist, data.frame(Position = rep(position, length(inv_breaks)-1), Bin = h$breaks[-1], Count = h$counts))
#  }
#
#  hchart(inv_hist, "areaspline", hcaes(x = Bin, y = Count, group = Position), stack = "overlap", marker = list(enabled = FALSE)) |>
#    hc_colors(selected_colors) |>
#    hc_title(text = "Distribution of Percent Invalid Barcode Positions") |>
#    hc_xAxis(title = list(text = "Percent Invalid"), type = "linear") |>
#    hc_yAxis(title = list(text = "Count")) |>
#    hc_tooltip(crosshairs = TRUE, animation = FALSE,
#      formatter = JS("function () {return '<b>' + this.series.name + '</b><br><b>' + this.x + '% invalid</b><br/>Count: <b>' + this.y + '</b>';}") 
#    ) |>
#    hc_exporting(enabled = T, filename = "invalid.barcodes.distribution",
#      buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
#  )
#} else {
#  print("Too few invalid barcodes to create a plot with, which is a very good thing!")
#}

:::

###
::: {.card}
This table details the number of invalid barcode positions as a 
percent of total linked-read barcodes within a sample.
It's likely that percentages will have greater diagnostic value than the total counts
because the number of reads and barcodes will vary between individual samples.

In [ ]:
column_description <- c("name of the sample", "total number of barcodes a sample has")
for(pos in position_names){
  column_description <- c(column_description, paste("number of invalidations at position", pos))
}

headerCallback <- c(
  "function(thead, data, start, end, display){",
  paste0(" var tooltips = ['", paste(column_description, collapse = "','"), "'];"),
  "  for(var i=0; i<tooltips.length; i++){",
  "    $('th:eq('+i+')',thead).attr('title', tooltips[i]);",
  "  }",
  "}"
)

DT::datatable(
  invalids_percent,
  rownames = FALSE,
  extensions = "Buttons",
  fillContainer = T,
  colnames = names(invalids_percent)[-1],
  caption = "Invalid positions relative to total barcodes",
  options = list(
    dom = "Brtp",
    buttons = c("csv"),
    scrollX = TRUE,
    headerCallback = JS(headerCallback)
  )
)

:::

# Per-Sample Invalidations
## Segment viz description
<h2> Per-Sample Barcode and Position Invalidations </h2>
These three visualizations show the occurrence of invalid barcodes.
The left figure shows this in finer detail as the percent invalid barcodes per 
individual. The rightmost figure shows the proportion of invalid barcode positions 
per individual. Together, these two figures should help you assess if any individuals 
have an unusually high amount of barcode invalidations and if any particular barcode 
position appears to be more prone to failure than others.

## validity plots
### {.flow}

In [ ]:
tb<-plotdata %>%
  pivot_longer(-Sample, names_to = "Barcode", values_to = "Count")

hchart(tb, "bar", hcaes(x = Sample, y = Count, group = Barcode), stacking = "percent") |>
  hc_colors(c("#707070", "#b88ace")) |>
  hc_title(text = "Valid vs Invalid Barcodes") |>
  hc_caption(text = "Given as percents") |>
  hc_xAxis(title = list(text = "")) |>
  hc_yAxis(title = list(text = "percent")) |>
  hc_tooltip(crosshairs = TRUE, animation = FALSE) |>
  hc_exporting(enabled = T, filename = "valid_barcodes.percent.per_sample",
    buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
)

### {.flow}

In [ ]:
invalids_long <- filter(invalids_long, Position != "position TotalBarcodes")
highchart() |>
  hc_chart(inverted=TRUE) |>
  hc_add_series(data = invalids_long,  type = "scatter", hcaes(x = Sample, y = Count, group = Position), marker = list(symbol = "circle")) |>
  hc_colors(selected_colors) |>
  hc_title(text = "Percent Invalid Barcode Positions") |>
  hc_xAxis(type = "category", title = list(text = ""), categories = unique(invalids_long$Sample)) |>
  hc_yAxis(title = list(text = "percent invalid"), min = 0, max = 100) |>
  hc_tooltip(crosshairs = TRUE, animation = FALSE,
    formatter = JS("function () {return '<b>' + this.x + '</b><br><b>' + this.series.name + '</b><br/>Percent: <b>' + this.y + '</b>';}") 
  ) |>
  hc_exporting(enabled = T, filename = "invalid.positions.per_sample",
    buttons = list(contextButton = list(menuItems = c("downloadPNG", "downloadJPEG", "downloadPDF", "downloadSVG")))
)